In [ ]:
# ===============================
# IMPORT REQUIRED LIBRARIES
# ===============================

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader

import numpy as np
import os
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

CLASS_NAMES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

In [ ]:
# ===============================
# CONFIGURATION
# ===============================

device = torch.device("cpu")

BASE_PATH = r"C:\Recommendation_Engine"

DATA_PATH = os.path.join(BASE_PATH, "data")
MODEL_PATH = os.path.join(BASE_PATH, "models", "fashion_model.pth")
FEATURE_PATH = os.path.join(BASE_PATH, "features", "features.npy")
LABEL_PATH = os.path.join(BASE_PATH, "features", "labels.npy")

os.makedirs(os.path.join(BASE_PATH, "models"), exist_ok=True)
os.makedirs(os.path.join(BASE_PATH, "features"), exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

# Set True only when you want to wipe checkpoints and retrain from scratch
FORCE_RETRAIN = False

# ImageNet normalization (matches utils/preprocessing.py and pretrained ResNet18)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

num_classes = 10
epochs_head = 8          # train classifier head only
epochs_finetune = 4      # then unfreeze last ResNet block
batch_size = 64

if FORCE_RETRAIN:
    for path in (MODEL_PATH, FEATURE_PATH, LABEL_PATH):
        if os.path.exists(path):
            os.remove(path)
            print(f"Removed stale file: {path}")

In [ ]:
# ===============================
# DATA LOADING & PREPROCESSING
# ===============================

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_dataset = torchvision.datasets.FashionMNIST(
    root=DATA_PATH,
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = torchvision.datasets.FashionMNIST(
    root=DATA_PATH,
    train=False,
    download=True,
    transform=eval_transform,
)

# Full training set for better embeddings and recommendations
feature_dataset = torchvision.datasets.FashionMNIST(
    root=DATA_PATH,
    train=True,
    download=False,
    transform=eval_transform,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
feature_loader = DataLoader(feature_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Training images: {len(train_dataset)}")
print(f"Catalog images: {len(feature_dataset)}")

In [ ]:
# ===============================
# MODEL SETUP (Pretrained ResNet18)
# ===============================

model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()


def train_one_epoch(loader, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss, 100 * correct / total

In [ ]:
# ===============================
# TRAIN MODEL (two-phase: head, then layer4)
# ===============================

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
    print("Model loaded. Skipping training.")
else:
    print("Phase 1: training classifier head...")
    optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

    for epoch in range(epochs_head):
        loss, train_acc = train_one_epoch(train_loader, optimizer)
        print(f"Head epoch {epoch + 1}/{epochs_head} - loss: {loss:.2f}, train acc: {train_acc:.2f}%")

    print("Phase 2: fine-tuning layer4 + head...")
    for param in model.layer4.parameters():
        param.requires_grad = True

    optimizer = optim.Adam([
        {"params": model.fc.parameters(), "lr": 1e-3},
        {"params": model.layer4.parameters(), "lr": 1e-4},
    ])

    for epoch in range(epochs_finetune):
        loss, train_acc = train_one_epoch(train_loader, optimizer)
        print(f"Finetune epoch {epoch + 1}/{epochs_finetune} - loss: {loss:.2f}, train acc: {train_acc:.2f}%")

    torch.save(model.state_dict(), MODEL_PATH)
    print("Model saved successfully.")

In [ ]:
# ===============================
# MODEL EVALUATION
# ===============================

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

In [ ]:
# ===============================
# FEATURE EXTRACTION (full catalog, L2-normalized)
# ===============================

if os.path.exists(FEATURE_PATH):
    features = np.load(FEATURE_PATH)
    labels_all = np.load(LABEL_PATH)
    print(f"Features loaded: {features.shape}")
else:
    print("Extracting features from full training catalog...")

    feature_extractor = nn.Sequential(*list(model.children())[:-1])
    feature_extractor.eval()

    features = []
    labels_all = []

    with torch.no_grad():
        for images, labels in feature_loader:
            images = images.to(device)
            output = feature_extractor(images)
            output = output.view(output.size(0), -1)
            features.append(output.cpu().numpy())
            labels_all.append(labels.numpy())

    features = np.vstack(features).astype(np.float32)
    labels_all = np.hstack(labels_all)

    # L2-normalize so cosine similarity reflects direction only
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    features = features / np.maximum(norms, 1e-8)

    np.save(FEATURE_PATH, features)
    np.save(LABEL_PATH, labels_all)
    print(f"Features saved: {features.shape}")

In [ ]:
# ===============================
# RECOMMENDATION FUNCTION
# ===============================

def recommend(index, top_k=5):
    """Return top-k visually similar catalog items by cosine similarity."""
    query_feature = features[index].reshape(1, -1)
    similarities = cosine_similarity(query_feature, features)[0].copy()

    # Skip self-match
    similarities[index] = -np.inf

    return np.argsort(similarities)[-top_k:][::-1]


def recommendation_hit_rate(n_samples=200, top_k=5):
    """Fraction of recommendations that share the query's class."""
    rng = np.random.default_rng(42)
    indices = rng.choice(len(features), size=min(n_samples, len(features)), replace=False)
    hits = 0
    for idx in indices:
        recs = recommend(idx, top_k=top_k)
        hits += np.isin(labels_all[recs], labels_all[idx]).sum()
    return 100 * hits / (len(indices) * top_k)

In [ ]:
sample_index = 10
recommended = recommend(sample_index)

print("Input index:", sample_index)
print("Query class:", CLASS_NAMES[labels_all[sample_index]])
print("Recommended indices:", recommended)
print("Recommended classes:", [CLASS_NAMES[labels_all[i]] for i in recommended])
print(f"Recommendation class hit rate (@5): {recommendation_hit_rate():.1f}%")

In [ ]:
# ===============================
# VISUALIZE RECOMMENDATIONS
# ===============================

visual_dataset = torchvision.datasets.FashionMNIST(
    root=DATA_PATH,
    train=True,
    download=False,
    transform=transforms.ToTensor(),
)

def show_recommendations(index):
    recommended = recommend(index)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 6, 1)
    plt.imshow(visual_dataset[index][0].squeeze(), cmap="gray")
    plt.title(f"Query\n{CLASS_NAMES[labels_all[index]]}")
    plt.axis("off")

    for i, rec_index in enumerate(recommended):
        plt.subplot(1, 6, i + 2)
        plt.imshow(visual_dataset[rec_index][0].squeeze(), cmap="gray")
        match = "match" if labels_all[rec_index] == labels_all[index] else "diff"
        plt.title(f"Rec {i + 1}\n{CLASS_NAMES[labels_all[rec_index]]} ({match})")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_recommendations(sample_index)